<!-- nav -->
[← 1 · Values and their domains](01_Values_And_Their_Domains.ipynb) · [Home](../../README.md) · [3 · The smallest expression, a literal →](03_The_Smallest_Expression_A_Literal.ipynb)

# 2 · One value, a domain in each language

Write `42` in Python, TypeScript, C and SystemVerilog, and each language gives it a type of its own: Python an `int`,
unbounded; TypeScript a `number`, a 64-bit float; C an `int`, as wide as the ABI it is compiled for makes it;
SystemVerilog an `integer`, 32 bits of four-state logic. Each of those types is a domain, stored data of the kinds
tutorial 1 showed, and each language's dialect owns its domains by the names its language gives them.

This notebook reads, for each language, the domain its standard gives `42`, by name. There's no term yet but C's: C
leaves `int` to the implementation, so a C term reads it through the ABI its expression's holder names. Tutorial 3
infers each domain from a literal.

The store is the toolkit's `dialects()`: every dialect installed and defined. `I` is where each dialect's singleton is
found, by its global name.

In [1]:
from mbse.Expressions.C import Expressions as C
from mbse.Expressions.Framework import Domains, Standards, Terms
from toolkit import dialects

store = dialects()
I = store.Instances.Expressions  # each dialect's singleton, which owns its domains by name
python, typescript = I.Python.Python3, I.TypeScript.TypeScript5


def kinds(dialect):
    """A dialect's domains by name, each with its kind: the dialect whose kind it is, and the kind's tag."""
    return {name: Domains.kind_of(domain)[:2] for name, domain in Domains.domains(dialect).items()}

<a id="T02.1"></a>
## 02.1 · Each language's own domains

`Domains.domains` gives a dialect's domains by name, in the order its `define` wrote them. Python3's are the built-in
types of the values Python's literals write:

In [2]:
kinds(python)

{'bool': ('isoiec11404', 'boolean'),
 'int': ('isoiec11404', 'integer'),
 'float': ('ieee754', 'ieee754'),
 'str': ('python3', 'str'),
 'bytes': ('isoiec11404', 'octetstring'),
 'NoneType': ('python3', 'nonetype'),
 'ellipsis': ('python3', 'ellipsis')}

Most are of a standard dialect's kind: `bool` is ISO/IEC 11404's `boolean`, `int` its `integer`, `bytes` its
`octetstring`, `float` IEEE 754's. Three are Python3's own. `str` is a sequence of code points, lone surrogates included
(`'\ud800'` is a Python string), so Python3 defines it rather than take Unicode's `string`. `NoneType` and `ellipsis`,
named as Python names `type(None)` and `type(...)`, are tokens: each is the domain of one value, `None` or `...`.

TypeScript5's tell the same story in TypeScript's terms: its `string` is its own, a sequence of UTF-16 code units, and
`null` is a token with a domain of its own:

In [3]:
assert kinds(python)["str"] == ("python3", "str") != kinds(I.Unicode.Expressions)["string"] == ("unicode", "unicode")
kinds(typescript)

{'boolean': ('isoiec11404', 'boolean'),
 'number': ('ieee754', 'ieee754'),
 'bigint': ('isoiec11404', 'integer'),
 'string': ('typescript5', 'string'),
 'null': ('typescript5', 'null')}

The editions each dialect covers are data too (tutorial 1, 01.7): Python3 covers Python 3.0 to 3.15, each a
version as the Python Software Foundation numbers it, and TypeScript5 TypeScript 5.0 to 5.9.

In [4]:
assert Standards.properties(Standards.find(python, "Python 3.12")) == {"bodies": ("org.python",), "revision": ("version", "3.12")}
list(Standards.standards(python)), list(Standards.standards(typescript))

(['Python 3.0',
  'Python 3.1',
  'Python 3.2',
  'Python 3.3',
  'Python 3.4',
  'Python 3.5',
  'Python 3.6',
  'Python 3.7',
  'Python 3.8',
  'Python 3.9',
  'Python 3.10',
  'Python 3.11',
  'Python 3.12',
  'Python 3.13',
  'Python 3.14',
  'Python 3.15'],
 ['TypeScript 5.0',
  'TypeScript 5.1',
  'TypeScript 5.2',
  'TypeScript 5.3',
  'TypeScript 5.4',
  'TypeScript 5.5',
  'TypeScript 5.6',
  'TypeScript 5.7',
  'TypeScript 5.8',
  'TypeScript 5.9'])

<a id="T02.2"></a>
## 02.2 · 42 in Python and TypeScript

```text
Python      42      int
TypeScript  42      number          42n     bigint
```

Python's `42` is an `int`, and Python3's `int` is ISO/IEC 11404's `integer`: signed, raising on overflow, and with no
`width`, `least` or `greatest`, since Python's ints are unbounded. `Domains.properties` gives a domain's parameters and
what follows from them, by name:

In [5]:
integer = Domains.domains(python)["int"]
Domains.kind_of(integer)[:2], Domains.properties(integer)

(('isoiec11404', 'integer'),
 {'signed': True, 'overflow': 'raise', 'ordered': True})

TypeScript's `42` is a `number`, IEEE 754's `binary64`: 53 bits of precision, so it holds every integer up to
2^53 exactly, 42 among them, but it is a float. Write `42n` and it is a `bigint`, 11404's `integer` again, unbounded,
with the same parameters as Python's `int`:

In [6]:
number, bigint = Domains.domains(typescript)["number"], Domains.domains(typescript)["bigint"]
assert Domains.kind_of(bigint)[:2] == Domains.kind_of(integer)[:2] and Domains.properties(bigint) == Domains.properties(integer)
Domains.kind_of(number)[:2], Domains.properties(number)

(('ieee754', 'ieee754'),
 {'format': 'binary64',
  'rounding': 'roundTiesToEven',
  'width': 64,
  'radix': 2,
  'precision': 53,
  'emax': 1023,
  'ordered': True})

TypeScript's `number` and Python's `float` are the same domain's twins, binary64 rounding ties to even;
Python's `int` and `float` are not, so `1` and `1.0` are values of different domains, which is why an expression never
coerces one into the other ([10.4](10_Answers_You_Can_Trust.ipynb#T10.4)).

In [7]:
assert Domains.properties(number) == Domains.properties(Domains.domains(python)["float"])
assert Domains.kind_of(integer)[:2] != Domains.kind_of(Domains.domains(python)["float"])[:2]

<a id="T02.3"></a>
## 02.3 · 42 in C, under an ABI

```text
C           42      int             (the ABI's: SystemV.AMD64 or Arm.AAPCS64)
```

C's standard fixes only some of its types: `bool`, the exact-width integers (`int32_t`), the `_FloatN` and decimal
floating types. Those are C's own domains, and `int` is not among them:

In [8]:
own = Domains.domains(I.C.Expressions)
assert "int32_t" in own and "int" not in own
list(own)

['bool',
 'int8_t',
 'int16_t',
 'int32_t',
 'int64_t',
 'uint8_t',
 'uint16_t',
 'uint32_t',
 'uint64_t',
 '_Float16',
 '_Float32',
 '_Float64',
 '_Float128',
 '_Decimal32',
 '_Decimal64',
 '_Decimal128',
 'nullptr_t']

How wide an `int` is, whether a plain `char` is signed, what `long double` holds: C leaves these to the
implementation, and an ABI fixes them. Each ABI is a dialect of its own, holding those types as stored domains:
`SystemV.AMD64`, x86-64's System V ABI, and `Arm.AAPCS64`, Arm's 64-bit procedure call standard. A C expression names
the ABI it is read under: its holder's `abi`, which this one writes in the builder's calls, around the literal `42`
(tutorial 3 explains holders and terms). The same `42` held twice, once under each ABI, and once under none:

In [9]:
amd64, aapcs64 = I.SystemV.AMD64, I.Arm.AAPCS64
forty_two = lambda u: u.literal(lambda r: r.value(lambda v: v.as_scalar(42)))  # the root's spec: the literal 42
with store.transaction():
    on_amd64 = store.Expressions.C.Expressions.Expression(lambda e: e.root(forty_two).abi([amd64]))
    on_aapcs64 = store.Expressions.C.Expressions.Expression(lambda e: e.root(forty_two).abi([aapcs64]))
    on_none = store.Expressions.C.Expressions.Expression(lambda e: e.root(forty_two))
assert [C.Text.ToText(held) for held in (on_amd64, on_aapcs64, on_none)] == ["42", "42", "42"]
assert [entry.abi for entry in on_amd64.abi] == [amd64] and list(on_none.abi) == []
[list(Standards.standards(entry.abi)) for held in (on_amd64, on_aapcs64) for entry in held.abi]

[['System V Application Binary Interface, AMD64 Architecture Processor Supplement'],
 ['Procedure Call Standard for the Arm 64-bit Architecture (AArch64)']]

A C term reads its types by name through `C.Domains.Types`: among C's own domains first, then among its holder's
ABI's. Read through each holder, `int` agrees: 11404's `integer`, 32 bits, signed, on both, each ABI's own domain:

In [10]:
x86, arm = C.Domains.Types(Terms.term(on_amd64)), C.Domains.Types(Terms.term(on_aapcs64))


def read(types, name):
    """The domain a C term reads `name` as, through its holder: its kind and its parameters."""
    domain = types.named(name).domain
    return Domains.kind_of(domain)[:2], Domains.properties(domain)


assert read(x86, "int") == read(arm, "int") and x86.named("int").domain is not arm.named("int").domain
read(x86, "int")

(('isoiec11404', 'integer'),
 {'width': 32,
  'signed': True,
  'overflow': 'raise',
  'least': -2147483648,
  'greatest': 2147483647,
  'ordered': True})

`char` differs. On x86-64 a plain `char` is signed, an `integer` of width 8 (-128 to 127); on AArch64 it is
unsigned, 11404's `modulo` 256 (0 to 255), wrapping:

In [11]:
read(x86, "char"), read(arm, "char")

((('isoiec11404', 'integer'),
  {'width': 8,
   'signed': True,
   'overflow': 'raise',
   'least': -128,
   'greatest': 127,
   'ordered': True}),
 (('isoiec11404', 'modulo'),
  {'modulus': 256, 'least': 0, 'greatest': 255, 'ordered': True}))

So does `long double`: on x86-64 it is x87's 80-bit extended format, which IEEE 754 allows but names no format
for, so the domain holds its parameters alone (radix 2, 64 digits of precision); on AArch64 it is `binary128`:

In [12]:
read(x86, "long double"), read(arm, "long double")

((('ieee754', 'ieee754'),
  {'rounding': 'roundTiesToEven',
   'width': 80,
   'radix': 2,
   'precision': 64,
   'emax': 16383,
   'ordered': True}),
 (('ieee754', 'ieee754'),
  {'format': 'binary128',
   'rounding': 'roundTiesToEven',
   'width': 128,
   'radix': 2,
   'precision': 113,
   'emax': 16383,
   'ordered': True}))

C's own types read the same through either holder, and through none; but a holder that names no ABI has no
`int`, and reading one is refused, by name:

In [13]:
unnamed = C.Domains.Types(Terms.term(on_none))
assert unnamed.named("int32_t").domain is x86.named("int32_t").domain is own["int32_t"]
try:
    unnamed.named("int")
except ValueError as error:
    print(error)

'int' is not a type C defines, and its expression names no ABI


C's editions, C11 to C23, are its dialect's, whichever ABI a holder names:

In [14]:
{designation: Standards.tags(edition)[1:] for designation, edition in Standards.standards(I.C.Expressions).items()}

{'ISO/IEC 9899:2011': ('C11',),
 'ISO/IEC 9899:2018': ('C17',),
 'ISO/IEC 9899:2024': ('C23',)}

<a id="T02.4"></a>
## 02.4 · 42 in SystemVerilog

```text
SystemVerilog  42   integer         int (2-state)
```

SystemVerilog's `42`, an unsized decimal number, is an `integer`: 32 bits, signed, and four-state, each bit `0`, `1`,
`x` or `z`, so its kind is SystemVerilog's own `logic`. `int` is not the same type: the two-state one, its bits only `0`
or `1`, of kind `bit`. Both have the same width and range; the kind tells them apart:

In [15]:
sv = Domains.domains(I.SystemVerilog.Expressions)
assert Domains.properties(sv["integer"]) == Domains.properties(sv["int"])
{name: Domains.kind_of(sv[name])[:2] for name in ("integer", "int")}, Domains.properties(sv["integer"])

({'integer': ('systemverilog', 'logic'), 'int': ('systemverilog', 'bit')},
 {'width': 32,
  'signed': True,
  'least': -2147483648,
  'greatest': 2147483647,
  'ordered': True})

Its other domains are of the same two kinds, `logic` (`reg`, `time`) and `bit` (`byte`, `shortint`,
`longint`), but for its reals, which are IEEE 754's: `real` is `binary64`, `shortreal` `binary32`:

In [16]:
assert Domains.properties(sv["real"]) == Domains.properties(number)
kinds(I.SystemVerilog.Expressions), Domains.properties(sv["shortreal"])["format"]

({'logic': ('systemverilog', 'logic'),
  'reg': ('systemverilog', 'logic'),
  'integer': ('systemverilog', 'logic'),
  'time': ('systemverilog', 'logic'),
  'bit': ('systemverilog', 'bit'),
  'byte': ('systemverilog', 'bit'),
  'shortint': ('systemverilog', 'bit'),
  'int': ('systemverilog', 'bit'),
  'longint': ('systemverilog', 'bit'),
  'real': ('ieee754', 'ieee754'),
  'shortreal': ('ieee754', 'ieee754')},
 'binary32')

The editions it covers run from Verilog, IEEE Std 1364, to SystemVerilog, IEEE Std 1800, each found by its
designation or its common name:

In [17]:
editions = Standards.standards(I.SystemVerilog.Expressions)
assert Standards.find(I.SystemVerilog.Expressions, "SystemVerilog-2023") is editions["IEEE Std 1800-2023"]
list(editions)

['IEEE Std 1364-1995',
 'IEEE Std 1364-2001',
 'IEEE Std 1364-2005',
 'IEEE Std 1800-2005',
 'IEEE Std 1800-2009',
 'IEEE Std 1800-2012',
 'IEEE Std 1800-2017',
 'IEEE Std 1800-2023']

<a id="T02.5"></a>
## 02.5 · One value, many domains

Every domain above, side by side: the domain each language gives `42`, read from the store, with C's through each of
its holders, and MATLAB's `double` and Excel's `number`, which are IEEE 754's `binary64` as TypeScript's `number` is.
`holds` reads, from a domain's parameters alone, whether 42 is one of its values: an integer domain's range, or the
integers a binary floating format holds exactly, up to its radix to the power of its precision.

In [18]:
def holds(properties, value):
    """Whether `value`, an integer, is a value of a domain with these parameters."""
    if "precision" in properties:  # an IEEE 754 format: integers up to radix ** precision are exact
        return abs(value) <= properties["radix"] ** properties["precision"]
    return properties.get("least", value) <= value <= properties.get("greatest", value)


def describe(properties):
    """A domain's width, signedness and range, as text."""
    width = str(properties.get("width", "unbounded"))
    signed = {True: "signed", False: "unsigned", None: "-"}[properties.get("signed")]
    if "precision" in properties:
        span = f"exact to {properties['radix']}^{properties['precision']}"
    elif "least" in properties:
        span = f"{properties['least']} to {properties['greatest']}"
    else:
        span = "unbounded"
    return width, signed, span


matlab, excel = Domains.domains(I.MATLAB.Expressions), Domains.domains(I.Excel.Expressions)
rows = [("Python", "42", Domains.domains(python)["int"]), ("TypeScript", "42", number), ("TypeScript", "42n", bigint),
        ("C, AMD64", "42", x86.named("int").domain), ("C, AAPCS64", "42", arm.named("int").domain),
        ("SystemVerilog", "42", sv["integer"]), ("SystemVerilog", "int'(42)", sv["int"]),
        ("MATLAB", "42", matlab["double"]), ("Excel", "=42", excel["number"])]
print(f"{'language':<14}{'text':<10}{'type':<9}{'kind':<26}{'width':<10}{'signed':<9}{'range':<26}holds 42")
for language, text, domain in rows:
    dialect, tag, _ = Domains.kind_of(domain)
    properties = Domains.properties(domain)
    kind = f"{dialect}.{tag}" + (f" {properties['format']}" if "format" in properties else "")
    width, signed, span = describe(properties)
    print(f"{language:<14}{text:<10}{Domains.name_of(domain):<9}{kind:<26}{width:<10}{signed:<9}{span:<26}"
          f"{'yes' if holds(properties, 42) else 'no'}")
assert all(holds(Domains.properties(domain), 42) for _, _, domain in rows)

language      text      type     kind                      width     signed   range                     holds 42
Python        42        int      isoiec11404.integer       unbounded signed   unbounded                 yes
TypeScript    42        number   ieee754.ieee754 binary64  64        -        exact to 2^53             yes
TypeScript    42n       bigint   isoiec11404.integer       unbounded signed   unbounded                 yes
C, AMD64      42        int      isoiec11404.integer       32        signed   -2147483648 to 2147483647 yes
C, AAPCS64    42        int      isoiec11404.integer       32        signed   -2147483648 to 2147483647 yes
SystemVerilog 42        integer  systemverilog.logic       32        signed   -2147483648 to 2147483647 yes
SystemVerilog int'(42)  int      systemverilog.bit         32        signed   -2147483648 to 2147483647 yes
MATLAB        42        double   ieee754.ieee754 binary64  64        -        exact to 2^53             yes
Excel         =42      

Nine domains, and 42 is a value of each, but they are not one domain: an unbounded integer, a binary64 float,
two 32-bit two's complement integers that agree because both ABIs say so, and SystemVerilog's four-state `integer` beside
its two-state `int`. A domain decides what arithmetic on `42` means: in Python `2147483647 + 1` is `2147483648`; in C,
under either ABI, it overflows `int`. Tutorial 3 gives `42` a term in each language and infers these domains from it;
tutorial 5 does the arithmetic.

<!-- nav -->
[← 1 · Values and their domains](01_Values_And_Their_Domains.ipynb) · [Home](../../README.md) · [3 · The smallest expression, a literal →](03_The_Smallest_Expression_A_Literal.ipynb)